<a href="https://colab.research.google.com/github/shuaibharoon86-oss/Portfolio/blob/main/price_prediction_model_1_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import io
import pandas as pd

print("📂 Upload your CSV file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"✅ Loaded: {filename}")
display(df_raw.head())

import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

df = df_raw.copy()

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

# Keep useful columns
df = df[[
    "date",
    "avg_farm_rate",
    "avg_doc",
    "egg_per_30doz"
]].dropna()

print("✅ Cleaned data:", df.shape)

d = df.copy()

# Target = next day price
d["target"] = d["avg_farm_rate"].shift(-1)

# Lag features
for lag in [1, 2, 3, 7, 14]:
    d[f"price_lag_{lag}"] = d["avg_farm_rate"].shift(lag)
    d[f"doc_lag_{lag}"]   = d["avg_doc"].shift(lag)

# Rolling stats
for w in [3, 7, 14]:
    d[f"roll_mean_{w}"] = d["avg_farm_rate"].rolling(w).mean()
    d[f"roll_std_{w}"]  = d["avg_farm_rate"].rolling(w).std()

# Calendar
d["day_of_week"] = d["date"].dt.dayofweek
d["month"] = d["date"].dt.month

d = d.dropna().reset_index(drop=True)

FEATURES = [c for c in d.columns if c not in ["date", "target"]]

print("✅ XGBoost features ready")

split = int(len(d) * 0.8)

train = d.iloc[:split]
test  = d.iloc[split:]

X_train = train[FEATURES]
y_train = train["target"]

X_test = test[FEATURES]
y_test = test["target"]

print("\n🏋️ Training XGBoost...")

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_preds)

print(f"✅ XGBoost MAE: {xgb_mae:.2f}")

features_lstm = df[[
    "avg_farm_rate",
    "avg_doc",
    "egg_per_30doz"
]]

scaler = MinMaxScaler()

scaled_data = scaler.fit_transform(features_lstm)

SEQ_LEN = 30

X_lstm = []
y_lstm = []

for i in range(SEQ_LEN, len(scaled_data)):
    X_lstm.append(scaled_data[i-SEQ_LEN:i])
    y_lstm.append(scaled_data[i][0])

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

split_lstm = int(len(X_lstm) * 0.8)

X_train_lstm = X_lstm[:split_lstm]
y_train_lstm = y_lstm[:split_lstm]

X_test_lstm = X_lstm[split_lstm:]
y_test_lstm = y_lstm[split_lstm:]

print("\n🏋️ Training LSTM...")

lstm_model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(SEQ_LEN, X_lstm.shape[2])),

    Dropout(0.2),

    LSTM(32),

    Dropout(0.2),

    Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

lstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=15,
    batch_size=32,
    validation_data=(X_test_lstm, y_test_lstm)
)

lstm_preds = lstm_model.predict(X_test_lstm)

preds_full = np.zeros((len(lstm_preds), 3))
preds_full[:, 0] = lstm_preds[:, 0]

y_full = np.zeros((len(y_test_lstm), 3))
y_full[:, 0] = y_test_lstm

lstm_preds_actual = scaler.inverse_transform(preds_full)[:, 0]
y_actual = scaler.inverse_transform(y_full)[:, 0]

lstm_mae = mean_absolute_error(y_actual, lstm_preds_actual)

print(f"✅ LSTM MAE: {lstm_mae:.2f}")

print("\n🔥 Creating Hybrid Predictions...")

min_len = min(len(xgb_preds), len(lstm_preds_actual))

xgb_final = xgb_preds[-min_len:]
lstm_final = lstm_preds_actual[-min_len:]
actual_final = y_test.values[-min_len:]

hybrid_preds = (
    0.6 * xgb_final +
    0.4 * lstm_final
)

hybrid_mae = mean_absolute_error(actual_final, hybrid_preds)
hybrid_rmse = np.sqrt(mean_squared_error(actual_final, hybrid_preds))

print("\n📊 FINAL RESULTS")
print("=" * 40)

print(f"XGBoost MAE : {xgb_mae:.2f}")
print(f"LSTM MAE    : {lstm_mae:.2f}")
print(f"HYBRID MAE  : {hybrid_mae:.2f}")

print(f"\n✅ Hybrid RMSE: {hybrid_rmse:.2f}")

def predict_future_hybrid(days=7):

    # ---------- XGBOOST ----------
    last_row = d.iloc[-1:].copy()

    xgb_future = []

    for _ in range(days):

        pred_xgb = xgb_model.predict(last_row[FEATURES])[0]

        xgb_future.append(pred_xgb)

        last_row["avg_farm_rate"] = pred_xgb

    # ---------- LSTM ----------
    temp = scaled_data[-SEQ_LEN:].copy()

    lstm_future = []

    for _ in range(days):

        x = temp.reshape(1, SEQ_LEN, temp.shape[1])

        pred = lstm_model.predict(x, verbose=0)[0][0]

        # drift control
        last_val = temp[-1][0]

        pred = max(pred, last_val - 0.03)
        pred = min(pred, last_val + 0.03)

        lstm_future.append(pred)

        new_row = temp[-1].copy()
        new_row[0] = pred

        temp = np.vstack([temp[1:], new_row])

    # inverse scale
    preds_full = np.zeros((len(lstm_future), 3))
    preds_full[:, 0] = lstm_future

    lstm_future_actual = scaler.inverse_transform(preds_full)[:, 0]

    hybrid = (
        0.6 * np.array(xgb_future) +
        0.4 * np.array(lstm_future_actual)
    )

    return hybrid

days = int(input("\nEnter number of future days to predict: "))

future_preds = predict_future_hybrid(days)

print("\n📊 Future Predictions")
print("=" * 40)

for i, val in enumerate(future_preds):
    print(f"Day +{i+1}: PKR {val:.1f}")

future_dates = pd.date_range(
    df["date"].iloc[-1],
    periods=days+1
)[1:]

plt.figure(figsize=(13,5))

plt.plot(
    df["date"],
    df["avg_farm_rate"],
    label="Historical"
)

plt.plot(
    future_dates,
    future_preds,
    '--',
    linewidth=2,
    label="Hybrid Forecast"
)

plt.title("Hybrid XGBoost + LSTM Forecast")
plt.xlabel("Date")
plt.ylabel("PKR/kg")
plt.legend()

plt.show()

📂 Upload your CSV file:


Saving agbro_broiler_prices_v4.csv to agbro_broiler_prices_v4.csv
✅ Loaded: agbro_broiler_prices_v4.csv


,date,day,lhr_doc,lhr_farm,lhr_open,lhr_close,egg_per_30doz,year,avg_farm_rate,avg_doc
0,1/1/2020,Wednesday,16.5,136.0,135.0,128.0,3810.0,2020,139.8,16.8
1,1/2/2020,Thursday,16.5,132.0,134.0,128.0,3840.0,2020,136.8,16.4
2,1/3/2020,Friday,13.5,132.0,130.0,126.0,3870.0,2020,137.4,13.4
3,1/4/2020,Saturday,13.5,132.0,128.0,126.0,3900.0,2020,135.2,13.0
4,1/6/2020,Monday,12.5,130.0,130.0,126.0,3900.0,2020,134.4,12.2


✅ Cleaned data: (1523, 4)
✅ XGBoost features ready

🏋️ Training XGBoost...
✅ XGBoost MAE: 9.16

🏋️ Training LSTM...
Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - loss: 0.0288 - val_loss: 0.0086
Epoch 2/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0091 - val_loss: 0.0063
Epoch 3/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0078 - val_loss: 0.0047
Epoch 4/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0073 - val_loss: 0.0043
Epoch 5/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0066 - val_loss: 0.0039
Epoch 6/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0056 - val_loss: 0.0038
Epoch 7/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0058 - val_loss: 0.0035
Epoch 8/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0056 - val_loss: 0.0044
Epoch 9/15
30/38 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0046